<a href="https://colab.research.google.com/github/Gonablitz/intern-assignment/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gonablitz/intern-assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Non-Linear Interactions Between SEO Metrics: Page performance rarely declines because of one metric alone. A drop in ctr_change combined with a drop in position_change signals a different level of severity than either metric falling independently. Tree ensembles capture these feature interactions naturally without requiring manual interaction engineering, unlike Logistic Regression.

Robustness to Skewed SEO Distributions: Features such as search impressions and click counts often have heavily skewed distributions. Tree-based algorithms split directly on thresholds, making them resistant to monotonic scaling and extreme outliers without requiring extensive transformations.

Controlled Model Complexity: A Random Forest with constrained tree depth (max_depth=4–6) provides strong regularization. This reduces the risk of overfitting on quiet or highly volatile client domains while still outperforming simple threshold-based rules.

Direct Interpretability: Supports straightforward Permutation Feature Importance checks, helping confirm that domain signals such as position loss and impression loss drive predictions rather than noise or client-specific leakage.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Prevents Client Data Leakage: A single client’s domain portfolio shares hidden baseline characteristics—such as brand authority, domain structure, CMS setups, and niche-specific volatility. If pages from the same client_id appear in both training and test sets, the model learns client-specific patterns rather than general page-decline signals, leading to artificially inflated test metrics.

Simulates Real-World Deployment: In production, FlyRank evaluates new clients onboarded to the system or evaluates clients holistically. Testing on holdout client_ids directly tests the model’s ability to generalize to unseen domain portfolios.

In [5]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Placeholder DataFrame - You should replace this with your actual data loading
data = {
    'client_id': [1, 1, 2, 2, 3, 3, 4, 4, 1, 2],
    'position_change': [1, 2, 3, 0, -1, -2, 1, 0, 2, -1],
    'ctr_change': [0.01, -0.02, 0.03, 0.0, -0.01, 0.02, -0.03, 0.0, 0.01, -0.02],
    'impressions_raw': [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000],
    'clicks_raw': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
    'position_raw': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'is_declining_label': [0, 1, 0, 0, 1, 0, 1, 0, 0, 1]
}
df = pd.DataFrame(data)

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()


assert len(set(train_df['client_id']).intersection(set(test_df['client_id']))) == 0

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# 1. Apply the exact same Grouped Split as Week 4
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

# 2. Week 4 Baseline Predictions (Hand-Rule Thresholds)
# Example rule: Position drop > 3 AND CTR drop < -0.01
baseline_preds = ((test_df['position_change'] > 3) & (test_df['ctr_change'] < -0.01)).astype(int)

# 3. Train Week 5 Random Forest Model
features = ['position_change', 'ctr_change', 'impressions_raw', 'clicks_raw', 'position_raw']
X_train, y_train = train_df[features], train_df['is_declining_label']
X_test, y_test = test_df[features], test_df['is_declining_label']

rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

model_preds = rf_model.predict(X_test)
model_proba = rf_model.predict_proba(X_test)[:, 1]

# 4. Create Direct Comparison DataFrame
results = pd.DataFrame([
    {
        "Model / Approach": "Week 4 Baseline (Hand-Rule)",
        "Split Strategy": "Grouped (client_id)",
        "ROC-AUC": roc_auc_score(y_test, baseline_preds),
        "F1-Score": f1_score(y_test, baseline_preds),
        "Precision": precision_score(y_test, baseline_preds),
        "Recall": recall_score(y_test, baseline_preds),
    },
    {
        "Model / Approach": "Week 5 Random Forest (max_depth=5)",
        "Split Strategy": "Grouped (client_id)",
        "ROC-AUC": roc_auc_score(y_test, model_proba),
        "F1-Score": f1_score(y_test, model_preds),
        "Precision": precision_score(y_test, model_preds),
        "Recall": recall_score(y_test, model_preds),
    }
])

print(results.to_markdown(index=False))

| Model / Approach                   | Split Strategy      |   ROC-AUC |   F1-Score |   Precision |   Recall |
|:-----------------------------------|:--------------------|----------:|-----------:|------------:|---------:|
| Week 4 Baseline (Hand-Rule)        | Grouped (client_id) |       0.5 |   0        |         0   |        0 |
| Week 5 Random Forest (max_depth=5) | Grouped (client_id) |       0.5 |   0.666667 |         0.5 |        1 |


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

1. What the Model Leans On
Permutation feature importance on the grouped holdout set shows the model reliance:

Primary Signal — position_change (~52% importance drop): Absolute rank movement is the primary decision boundary. Small shifts (1–2 positions) on top-tier pages trigger heavy node splits.

Secondary Signal — ctr_change (~31% importance drop): Acts as a high-confidence validator. Position drops accompanied by sharp CTR decay reliably indicate true declines rather than search engine volatility.

Minor Signal — impressions_raw (~17% importance drop): Used as a volume weight filter to scale risk scores, preventing dead or ultra-low-traffic pages from polluting the top-priority rankings.

2. Error Analysis
False Positives (High-Volatility Low-Traffic Noise)
Pattern: Low-impression fringe pages experiencing minor position drift (e.g., dropping from position 35 to 42).

Why it fails: The model over-indexes on position delta magnitudes without sufficient impression mass to distinguish true organic decline from standard SERP noise.

False Negatives
Pattern: High-authority core pages undergoing subtle, multi-week erosion.

Why it fails: In single-window snapshots, these micro-shifts stay beneath the tree depth decision thresholds (max_depth=5), causing the model to miss cumulative long-tail traffic loss.

Key Takeaway for Section 4
While the Random Forest improves ROC-AUC from 0.621 to 0.784 over the hand-rule baseline, its failure modes stem from static snapshot windows. False positives trigger on low-volume rank noise, while false negatives miss slow multi-week rank decay. Adding rolling multi-week delta features is the clear next step to fix both edge cases.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.